# 4.2. Матричные операции как преобразования

Аналитика 101 · блок «Дата аналитика» · тема 04 «Математика для данных» · страница 4.2.

Здесь собраны шесть примеров из урока. Для расчётов используется таблица `data/clients/clients.csv` с данными о 1 000 клиентах учебного магазина.

Выполните ячейки по порядку. Последние три примера используют общую таблицу `df`, которую вы загрузите в первом из них.

Результаты в уроке получены на Python 3.13 с NumPy 2.5 и pandas 3.0. В более ранних версиях NumPy вещественные собственные значения могут выводиться как `[2. 1.]` вместо `[2.+0.j 1.+0.j]`. При решении системы с матрицей Гильберта величины ошибки и невязки зависят от реализации линейной алгебры. Ячейка отдельно проверяет, что найденный через `solve` вектор удовлетворяет системе с учётом округления.

## Где лежат данные

Распакуйте общий набор `data/` рядом с папкой урока. При запуске первая ячейка найдёт каталог, в котором находится `data/clients/`, и сделает его рабочим. Тогда все примеры смогут читать один и тот же CSV-файл.

In [1]:
import os
import pathlib
import platform

import numpy as np
import pandas as pd

root = pathlib.Path.cwd()
while not (root / "data" / "clients").is_dir() and root != root.parent:
    root = root.parent                      # поднимаемся, пока рядом не окажется data/clients/
assert (root / "data" / "clients" / "clients.csv").exists(), \
    "Не найдена папка data/clients/. Скачайте общий набор темы 04 и положите data/ рядом с папкой страницы"
os.chdir(root)

print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

Python 3.13.13 | numpy 2.5.0 | pandas 3.0.5


## Порядок матриц, их размеры и решение систем уравнений

### Когда важен порядок матриц и какие размерности совместимы

In [2]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

S = np.array([[2.0, 0.0], [0.0, 1.0]])     # растяжение по горизонтали
R = np.array([[0.8, -0.6], [0.6, 0.8]])    # поворот на 36,87°
print("S @ R — сначала поворот, потом растяжение:\n", S @ R)
print("R @ S — сначала растяжение, потом поворот:\n", R @ S)
print("det:", np.linalg.det(S @ R).round(3), np.linalg.det(R @ S).round(3))

A = np.ones((3, 4))
B = np.ones((3, 4))
print("A * B:", (A * B).shape, "  A @ B.T:", (A @ B.T).shape, "  A.T @ B:", (A.T @ B).shape)
try:
    A @ B
except ValueError as e:
    print("A @ B → ValueError:", e)

S @ R — сначала поворот, потом растяжение:
 [[ 1.6 -1.2]
 [ 0.6  0.8]]
R @ S — сначала растяжение, потом поворот:
 [[ 1.6 -0.6]
 [ 1.2  0.8]]
det: 2.0 2.0
A * B: (3, 4)   A @ B.T: (3, 3)   A.T @ B: (4, 4)
A @ B → ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 3 is different from 4)


### Погрешность двух способов решить систему

In [3]:
import numpy as np
import pandas as pd

n = 10
H = 1 / (np.arange(1, n + 1)[:, None] + np.arange(n))   # матрица Гильберта 10 × 10
x_true = np.ones(n)
b = H @ x_true                                           # знаем ответ заранее: x = (1, …, 1)
print(f"cond(H) = {np.linalg.cond(H):.1e}")
x_solve = np.linalg.solve(H, b)
for name, x in [("inv(H) @ b ", np.linalg.inv(H) @ b), ("solve(H, b)", x_solve)]:
    print(f"{name}  ошибка в x: {np.abs(x - x_true).max():.1e}   невязка H·x − b: {np.abs(H @ x - b).max():.1e}")
np.testing.assert_allclose(H @ x_solve, b, rtol=1e-12, atol=1e-14)

df = pd.read_csv("data/clients/clients.csv")
X = np.c_[np.ones(len(df)), df["income"], df["orders"]]
Z = np.c_[np.ones(len(df)), (df[["income", "orders"]] - df[["income", "orders"]].mean()) / df[["income", "orders"]].std(ddof=0)]
print(f"рубли:          cond(X) = {np.linalg.cond(X):.1e}   cond(XᵀX) = {np.linalg.cond(X.T @ X):.1e}")
print(f"стандартизация: cond(X) = {np.linalg.cond(Z):.1f}       cond(XᵀX) = {np.linalg.cond(Z.T @ Z):.1f}")

cond(H) = 1.6e+13
inv(H) @ b   ошибка в x: 5.5e-03   невязка H·x − b: 2.5e-05
solve(H, b)  ошибка в x: 1.7e-04   невязка H·x − b: 4.4e-16
рубли:          cond(X) = 2.2e+05   cond(XᵀX) = 5.0e+10
стандартизация: cond(X) = 1.6       cond(XᵀX) = 2.4


В этом запуске solve даёт меньшую ошибку решения и меньшую невязку. Невязка показывает, насколько H·x отличается от b. Числа при повторном запуске зависят от библиотеки линейной алгебры, которую использует NumPy.

### Ранг при коллинеарных признаках

In [4]:
import numpy as np
import pandas as pd
np.set_printoptions(precision=4, suppress=True)

df = pd.read_csv("data/clients/clients.csv")
y = df["spend"].to_numpy(dtype=float)
X = np.c_[np.ones(len(df)), df["income"], df["orders"], df["income"] / 1000]   # доход ещё раз, в тысячах
print("столбцов:", X.shape[1], "  ранг:", np.linalg.matrix_rank(X))
print("det(XᵀX) =", np.linalg.det(X.T @ X))
try:
    np.linalg.solve(X.T @ X, X.T @ y)
except np.linalg.LinAlgError as e:
    print("solve → LinAlgError:", e)
w, rss, rank, sv = np.linalg.lstsq(X, y)
print("lstsq: ранг", rank, "  сингулярные числа:", sv.round(1))
print("w =", w, "  вклад дохода:", round(w[1] + w[3] / 1000, 4))

столбцов: 4   ранг: 3
det(XᵀX) = 0.0
solve → LinAlgError: Singular matrix
lstsq: ранг 3   сингулярные числа: [2784574.8      68.9      12.4       0. ]
w = [ 204.6891    0.0347 2549.5291    0.    ]   вклад дохода: 0.0347


Коэффициент при доходе в тысячах рублей равен примерно 3,5 · 10⁻⁵ и округляется до нуля на экране. Из всех решений lstsq выбрала вектор с наименьшей евклидовой нормой. Последнее сингулярное число близко к нулю, ранг равен 3. Из-за округления в другой сборке solve иногда возвращает коэффициенты. Их наличие ещё не означает, что решение единственно.

## Облако точек, главные компоненты и прогноз трат

В первой ячейке загрузим таблицу и выберем 30 клиентов для рисунка. В следующих двух воспользуемся данными всей тысячи клиентов.

### Как пять матриц преобразуют точки

In [5]:
import numpy as np
import pandas as pd
np.set_printoptions(precision=3, suppress=True)

df = pd.read_csv("data/clients/clients.csv")
X = df[["income", "spend"]].to_numpy(dtype=float)
Z = (X - X.mean(axis=0)) / X.std(axis=0)        # у каждого столбца среднее 0 и стандартное отклонение 1
idx = np.random.default_rng(58).choice(len(Z), 30, replace=False)
P = Z[idx]                                      # две координаты каждого из 30 клиентов на рисунке

M = {"растяжение":  np.array([[2.0, 0.0], [0.0, 1.0]]),
     "сдвиг":       np.array([[1.0, 0.8], [0.0, 1.0]]),
     "поворот":     np.array([[0.8, -0.6], [0.6, 0.8]]),
     "общая":       np.array([[1.5, 1.0], [0.5, 1.0]]),
     "вырожденная": np.array([[1.0, 2.0], [0.5, 1.0]])}

for name, A in M.items():
    Q = P @ A.T                                 # строка результата хранит координаты вектора A @ p
    lam = np.linalg.eigvals(A)
    print(f"{name:11} det = {np.linalg.det(A):4.2f}  ранг облака "
          f"{np.linalg.matrix_rank(P)} → {np.linalg.matrix_rank(Q)}  λ = {np.round(lam, 2)}")

A = M["общая"]
lam, V = np.linalg.eig(A)                       # столбцы V — собственные векторы длины 1
lam, V = lam.real, V.real                       # для этой матрицы мнимые части равны нулю
print("A @ v₁ =", A @ V[:, 0], "  λ₁ · v₁ =", lam[0] * V[:, 0])
print("v₁ ∥", V[:, 0] / V[1, 0], "  v₂ ∥", V[:, 1] / V[0, 1])

растяжение  det = 2.00  ранг облака 2 → 2  λ = [2.+0.j 1.+0.j]
сдвиг       det = 1.00  ранг облака 2 → 2  λ = [1.+0.j 1.+0.j]
поворот     det = 1.00  ранг облака 2 → 2  λ = [0.8+0.6j 0.8-0.6j]
общая       det = 1.00  ранг облака 2 → 2  λ = [2. +0.j 0.5+0.j]
вырожденная det = 0.00  ранг облака 2 → 1  λ = [2.+0.j 0.+0.j]
A @ v₁ = [1.789 0.894]   λ₁ · v₁ = [1.789 0.894]
v₁ ∥ [2. 1.]   v₂ ∥ [ 1. -1.]


В выводе NumPy 2.5 запись 2.+0.j означает вещественное значение 2 с нулевой мнимой частью. В более ранних версиях его можно увидеть как 2. У матрицы поворота на 37° оба значения имеют ненулевую мнимую часть (0,8 ± 0,6i). У матрицы сдвига одно и то же значение 1 встречается дважды, а все собственные векторы лежат на одной прямой.

### Сколько дисперсии сохраняет первая главная компонента

In [6]:
cols = ["age", "income", "orders", "spend"]
X = df[cols].to_numpy(dtype=float)              # (1000, 4)
Z = (X - X.mean(axis=0)) / X.std(axis=0)
C = Z.T @ Z / len(Z)                            # ковариационная матрица, (4, 4)
print(C)
print("совпадает с np.cov:", np.allclose(C, np.cov(Z, rowvar=False, ddof=0)))

lam, V = np.linalg.eig(C)
lam, V = lam.real, V.real                       # у симметричной матрицы всё вещественное
order = np.argsort(lam)[::-1]                   # по убыванию λ
lam, V = lam[order], V[:, order]
print("λ:", lam, " сумма:", lam.sum().round(3))
print("доля разброса:", (lam / lam.sum()).round(3))
print("v₁:", "  ".join(f"{c} {v:.3f}" for c, v in zip(cols, V[:, 0])))
pc1 = Z @ V[:, 0]                               # координата каждого клиента вдоль v₁
print("дисперсия вдоль v₁:", pc1.var().round(3))

[[1.    0.146 0.08  0.04 ]
 [0.146 1.    0.418 0.5  ]
 [0.08  0.418 1.    0.836]
 [0.04  0.5   0.836 1.   ]]
совпадает с np.cov: True
λ: [2.208 1.005 0.631 0.156]  сумма: 4.0
доля разброса: [0.552 0.251 0.158 0.039]
v₁: age 0.119  income 0.482  orders 0.605  spend 0.622
дисперсия вдоль v₁: 2.208


Таблица df загружается в предыдущей ячейке. Для вещественных симметричных матриц можно также вызвать eigh. Она возвращает вещественные собственные значения в порядке возрастания.

### Прогноз трат по доходу и числу заказов

In [7]:
np.set_printoptions(precision=4, suppress=True)
X = np.c_[np.ones(len(df)), df["income"], df["orders"]]   # (1000, 3): единицы — свободный член
y = df["spend"].to_numpy(dtype=float)                       # (1000,)

w, rss, rank, sv = np.linalg.lstsq(X, y)
print("w =", w, " ранг X:", rank)

w_ne = np.linalg.solve(X.T @ X, X.T @ y)       # нормальное уравнение XᵀX w = Xᵀy
print("через нормальное уравнение:", w_ne, np.allclose(w, w_ne))

pred = X @ w                                    # (1000, 3) @ (3,) → (1000,)
r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
print("R² =", round(r2, 3))
print("клиент 1:", df.loc[0, ["income", "orders", "spend"]].tolist(), "→ прогноз", round(pred[0]))
print("Xᵀ(y − Xw) / n =", X.T @ (y - pred) / len(y))   # остатки перпендикулярны столбцам X

w = [ 204.6891    0.0347 2549.5291]  ранг X: 3
через нормальное уравнение: [ 204.6891    0.0347 2549.5291] True
R² = 0.727
клиент 1: [80000, 2, 9080] → прогноз 8083
Xᵀ(y − Xw) / n = [0. 0. 0.]


Последняя строка показывает скалярные произведения вектора остатков со столбцами X, делённые на число наблюдений. С точностью до округления все они равны нулю, как и следует из нормального уравнения.

## Проверка своих матриц

В файле `deform.py` можно задать четыре элемента матрицы 2 × 2 и вычислить её определитель, ранг и собственные значения. Для вещественных собственных направлений скрипт выводит пояснение или координаты векторов. Сравните результат с тем, что показывает интерактив для той же матрицы.